# Machine Learning multimodel amb AV45

Aquesta notebook segueix una idea semblant a la pràctica del Titanic: comparar diversos models clàssics de *Machine Learning*, fer una cerca controlada d'hiperparàmetres i seleccionar llindars de decisió segons F1 i F2.

Flux general:

1. Carregar el dataset AV45 preprocessat.
2. Fer una divisió per pacient en entrenament, validació i test.
3. Reduir espacialment els volums PET i vectoritzar-los.
4. Aplicar `StandardScaler` i PCA ajustats només amb el conjunt d'entrenament.
5. Comparar diversos classificadors base.
6. Fer una cerca d'hiperparàmetres controlada.
7. Seleccionar els llindars òptims en validació segons F1 i F2.
8. Avaluar els millors models una única vegada sobre el conjunt de test.

> El conjunt de test no s'utilitza per seleccionar hiperparàmetres ni llindars. Només s'utilitza al final per avaluar el rendiment final.


In [ ]:
# ==========================================
# 1. Imports i configuració general
# ==========================================

import os
import random
import itertools
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import zoom

from sklearn.model_selection import GroupShuffleSplit, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

# Models opcionals
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as e:
    print("XGBoost no disponible:", e)
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except Exception as e:
    print("LightGBM no disponible:", e)
    LIGHTGBM_AVAILABLE = False


SEED = 1672891
random.seed(SEED)
np.random.seed(SEED)

# Canvia aquesta ruta si tens el dataset en una altra carpeta
BASE_DIR = r"D:\Universidad\TFG\OASIS"

X_PATH = os.path.join(BASE_DIR, "X_av45_128x128x64.npy")
Y_PATH = os.path.join(BASE_DIR, "y_av45_128x128x64.npy")
META_PATH = os.path.join(BASE_DIR, "metadata_av45_128x128x64.csv")

ML_SHAPE = (32, 32, 32)
VARIANZA_OBJETIVO_PCA = 0.90

CACHE_FEATURES_PATH = os.path.join(BASE_DIR, f"X_av45_ml_{ML_SHAPE[0]}x{ML_SHAPE[1]}x{ML_SHAPE[2]}.npy")

print("Configuració carregada")
print("XGBoost disponible:", XGBOOST_AVAILABLE)
print("LightGBM disponible:", LIGHTGBM_AVAILABLE)


In [ ]:
# ==========================================
# 2. Càrrega de dades
# ==========================================

X = np.load(X_PATH)
y = np.load(Y_PATH)
df = pd.read_csv(META_PATH)

print("Shape original de X:", X.shape)
print("Shape de y:", y.shape)
print("Shape de metadata:", df.shape)

display(df.head())

# Si X té canal explícit, l'eliminem per treballar amb volums 3D.
if X.ndim == 5:
    X = X[:, 0]

assert len(X) == len(y) == len(df), "X, y i metadata no tenen la mateixa longitud"

print("Shape de X sense canal:", X.shape)
print("\nDistribució global de classes:")
print(pd.Series(y).value_counts().sort_index())
print(pd.Series(y).value_counts(normalize=True).sort_index())


In [ ]:
# ==========================================
# 3. Neteja defensiva de NaN i infinits
# ==========================================

nan_per_img = np.isnan(X).sum(axis=(1, 2, 3))
inf_per_img = np.isinf(X).sum(axis=(1, 2, 3))

mask_ok = (nan_per_img == 0) & (inf_per_img == 0)

print("Imatges amb NaN/Inf eliminades:", np.sum(~mask_ok))

X = X[mask_ok]
y = y[mask_ok]
df = df.loc[mask_ok].reset_index(drop=True)

print("X final:", X.shape)
print("y final:", y.shape)
print("metadata final:", df.shape)


In [ ]:
# ==========================================
# 4. Divisió train / validació / test per pacient
# ==========================================

groups = df["OASISID"].values

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
trainval_idx, test_idx = next(gss_test.split(X, y, groups=groups))

X_trainval = X[trainval_idx]
X_test = X[test_idx]
y_trainval = y[trainval_idx]
y_test = y[test_idx]
groups_trainval = groups[trainval_idx]
groups_test = groups[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx_rel, val_idx_rel = next(gss_val.split(X_trainval, y_trainval, groups=groups_trainval))

X_train = X_trainval[train_idx_rel]
X_val = X_trainval[val_idx_rel]
y_train = y_trainval[train_idx_rel]
y_val = y_trainval[val_idx_rel]
groups_train = groups_trainval[train_idx_rel]
groups_val = groups_trainval[val_idx_rel]

print("Train:", X_train.shape, pd.Series(y_train).value_counts().sort_index().to_dict())
print("Validació:", X_val.shape, pd.Series(y_val).value_counts().sort_index().to_dict())
print("Test:", X_test.shape, pd.Series(y_test).value_counts().sort_index().to_dict())

print("\nPacients únics:")
print("Train:", len(np.unique(groups_train)))
print("Validació:", len(np.unique(groups_val)))
print("Test:", len(np.unique(groups_test)))

# Comprovació de no solapament de pacients entre particions
assert set(groups_train).isdisjoint(set(groups_val))
assert set(groups_train).isdisjoint(set(groups_test))
assert set(groups_val).isdisjoint(set(groups_test))

print("\nNo hi ha solapament de pacients entre train, validació i test.")


In [ ]:
# ==========================================
# 5. Reducció espacial i vectorització
# ==========================================

def resize_volume(volume, target_shape=ML_SHAPE):
    """
    Redimensiona un volum 3D a una mida objectiu.

    Paràmetres
    ----------
    volume : np.ndarray
        Volum PET tridimensional.
    target_shape : tuple
        Mida final desitjada del volum.

    Retorna
    -------
    np.ndarray
        Volum redimensionat en format float32.
    """
    factors = [t / s for t, s in zip(target_shape, volume.shape)]
    return zoom(volume, factors, order=1).astype(np.float32)


def preparar_features_ml(X_volumes, target_shape=ML_SHAPE):
    """
    Redueix espacialment els volums PET i els aplana per obtenir una matriu de característiques.

    Aquesta funció transforma cada volum 3D en un vector 1D, de manera que pugui ser utilitzat
    per models clàssics de Machine Learning.

    Paràmetres
    ----------
    X_volumes : np.ndarray
        Conjunt de volums PET.
    target_shape : tuple
        Mida a la qual es redimensiona cada volum.

    Retorna
    -------
    np.ndarray
        Matriu de característiques amb shape (n_mostres, n_voxels_redimensionats).
    """
    X_small = np.empty((len(X_volumes), *target_shape), dtype=np.float32)

    for i, vol in enumerate(X_volumes):
        if i % 50 == 0:
            print(f"Processant {i}/{len(X_volumes)}")
        X_small[i] = resize_volume(vol, target_shape)

    return X_small.reshape(len(X_small), -1)


if os.path.exists(CACHE_FEATURES_PATH):
    print("Carregant característiques cachejades:", CACHE_FEATURES_PATH)
    X_flat_all = np.load(CACHE_FEATURES_PATH)

    # Aplicar la mateixa divisió ja calculada
    X_train_flat = X_flat_all[trainval_idx][train_idx_rel]
    X_val_flat = X_flat_all[trainval_idx][val_idx_rel]
    X_test_flat = X_flat_all[test_idx]
else:
    print("Preparant característiques des de zero...")
    X_flat_all = preparar_features_ml(X, ML_SHAPE)
    np.save(CACHE_FEATURES_PATH, X_flat_all)
    print("Característiques guardades a:", CACHE_FEATURES_PATH)

    X_train_flat = X_flat_all[trainval_idx][train_idx_rel]
    X_val_flat = X_flat_all[trainval_idx][val_idx_rel]
    X_test_flat = X_flat_all[test_idx]

print("X_train_flat:", X_train_flat.shape)
print("X_val_flat:", X_val_flat.shape)
print("X_test_flat:", X_test_flat.shape)


In [ ]:
# ==========================================
# 6. StandardScaler + PCA ajustats només amb train
# ==========================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_flat)
X_val_scaled = scaler.transform(X_val_flat)
X_test_scaled = scaler.transform(X_test_flat)

X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_val_scaled = np.nan_to_num(X_val_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

pca = PCA(
    n_components=VARIANZA_OBJETIVO_PCA,
    svd_solver="full",
    random_state=SEED
)

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

N_COMPONENTS_PCA = X_train_pca.shape[1]
varianza_acumulada = np.cumsum(pca.explained_variance_ratio_)

print("Components PCA seleccionades:", N_COMPONENTS_PCA)
print("Variància explicada acumulada:", varianza_acumulada[-1])

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, N_COMPONENTS_PCA + 1), varianza_acumulada, marker="o")
plt.axvline(N_COMPONENTS_PCA, linestyle="--", color="red")
plt.xlabel("Components PCA")
plt.ylabel("Variància acumulada")
plt.grid(True)
plt.tight_layout()
plt.show()


## Funcions auxiliars

Aquestes funcions serveixen per calcular probabilitats, mètriques, llindars òptims i matrius de confusió. La classe positiva és la classe `1`, corresponent als subjectes amb Alzheimer segons l'etiqueta definida.


In [ ]:
# ==========================================
# 7. Funcions auxiliars de mètriques i llindars
# ==========================================

def get_scores_model(model, X_data):
    """
    Retorna la puntuació o probabilitat associada a la classe positiva.

    Si el model implementa `predict_proba`, s'utilitza la probabilitat de la classe 1.
    Si no, s'utilitza `decision_function` i es normalitza a l'interval [0, 1].

    Paràmetres
    ----------
    model : object
        Model de classificació entrenat.
    X_data : np.ndarray
        Matriu de característiques.

    Retorna
    -------
    np.ndarray
        Scores associats a la classe positiva.
    """
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]

    if hasattr(model, "decision_function"):
        raw = model.decision_function(X_data)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)

    return model.predict(X_data)


def metricas_binarias(y_true, preds, prefix=""):
    """
    Calcula mètriques de classificació binària centrades en la classe positiva.

    Paràmetres
    ----------
    y_true : np.ndarray
        Etiquetes reals.
    preds : np.ndarray
        Prediccions binàries.
    prefix : str
        Prefix opcional per als noms de les mètriques.

    Retorna
    -------
    dict
        Diccionari amb accuracy, precision, recall, F1, F2 i valors de la matriu de confusió.
    """
    cm = confusion_matrix(y_true, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        f"{prefix}accuracy": accuracy_score(y_true, preds),
        f"{prefix}precision_alzheimer": precision_score(y_true, preds, zero_division=0),
        f"{prefix}recall_alzheimer": recall_score(y_true, preds, zero_division=0),
        f"{prefix}f1_alzheimer": f1_score(y_true, preds, zero_division=0),
        f"{prefix}f2_alzheimer": fbeta_score(y_true, preds, beta=2, zero_division=0),
        f"{prefix}TN": tn,
        f"{prefix}FP": fp,
        f"{prefix}FN": fn,
        f"{prefix}TP": tp,
    }


def tabla_thresholds_validacion(y_true, scores, thresholds):
    """
    Avalua diferents llindars de decisió sobre el conjunt de validació.

    Aquesta funció no s'ha d'aplicar sobre el conjunt de test per seleccionar el llindar.

    Paràmetres
    ----------
    y_true : np.ndarray
        Etiquetes reals de validació.
    scores : np.ndarray
        Probabilitats o scores de la classe positiva.
    thresholds : iterable
        Valors de llindar a provar.

    Retorna
    -------
    pd.DataFrame
        Taula amb les mètriques obtingudes per a cada llindar.
    """
    rows = []

    for t in thresholds:
        preds = (scores >= t).astype(int)
        row = {"threshold": float(t)}
        row.update(metricas_binarias(y_true, preds, prefix="val_"))
        row["positius_predits"] = int(preds.sum())
        rows.append(row)

    return pd.DataFrame(rows)


def elegir_threshold(df_thresholds, criterio):
    """
    Selecciona el millor llindar segons una mètrica de validació.

    En cas d'empat, es prioritza un recall més alt i després una precisió més alta.

    Paràmetres
    ----------
    df_thresholds : pd.DataFrame
        Taula de mètriques per llindar.
    criterio : str
        Columna que es vol maximitzar, per exemple `val_f1_alzheimer` o `val_f2_alzheimer`.

    Retorna
    -------
    tuple
        Millor llindar i fila completa de resultats.
    """
    orden = df_thresholds.sort_values(
        by=[criterio, "val_recall_alzheimer", "val_precision_alzheimer"],
        ascending=False
    )
    best_row = orden.iloc[0]
    return float(best_row["threshold"]), best_row


def entrenar_y_evaluar_thresholds(model, X_train_data, y_train_data, X_val_data, y_val_data, thresholds):
    """
    Entrena un model i selecciona dos llindars en validació: un segons F1 i un segons F2.

    Paràmetres
    ----------
    model : object
        Model de classificació no entrenat.
    X_train_data : np.ndarray
        Característiques d'entrenament.
    y_train_data : np.ndarray
        Etiquetes d'entrenament.
    X_val_data : np.ndarray
        Característiques de validació.
    y_val_data : np.ndarray
        Etiquetes de validació.
    thresholds : iterable
        Llindars a provar.

    Retorna
    -------
    dict
        Diccionari amb el model entrenat, scores de validació, taula de llindars i millors llindars.
    """
    model.fit(X_train_data, y_train_data)

    scores_val = get_scores_model(model, X_val_data)
    df_thr = tabla_thresholds_validacion(y_val_data, scores_val, thresholds)

    best_t_f1, best_row_f1 = elegir_threshold(df_thr, "val_f1_alzheimer")
    best_t_f2, best_row_f2 = elegir_threshold(df_thr, "val_f2_alzheimer")

    return {
        "model": model,
        "scores_val": scores_val,
        "df_thresholds": df_thr,
        "best_t_f1": best_t_f1,
        "best_t_f2": best_t_f2,
        "best_row_f1": best_row_f1,
        "best_row_f2": best_row_f2,
    }


def plot_confusion_catala(y_true, preds, title):
    """
    Mostra una matriu de confusió amb etiquetes en català.

    Paràmetres
    ----------
    y_true : np.ndarray
        Etiquetes reals.
    preds : np.ndarray
        Prediccions binàries del model.
    title : str
        Títol de la figura.
    """
    cm = confusion_matrix(y_true, preds, labels=[0, 1])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Sa", "Alzheimer"]
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(cmap="Blues", colorbar=False, values_format="d", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicció")
    ax.set_ylabel("Etiqueta real")
    plt.tight_layout()
    plt.show()


## Comparació inicial de models

Primer es comparen diferents classificadors amb paràmetres base. L'objectiu és tenir una visió inicial del comportament de cada model abans de fer una cerca d'hiperparàmetres.


In [ ]:
# ==========================================
# 8. Comparació inicial de models base
# ==========================================

THRESHOLDS = np.arange(0.05, 0.95, 0.01)

n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale_pos_weight = n_neg / max(n_pos, 1)

models_base = {
    "Logistic Regression": LogisticRegression(
        random_state=SEED,
        class_weight="balanced",
        max_iter=5000
    ),
    "SVM RBF": SVC(
        random_state=SEED,
        class_weight="balanced",
        probability=True
    ),
    "Random Forest": RandomForestClassifier(
        random_state=SEED,
        class_weight="balanced",
        n_estimators=500,
        min_samples_leaf=2,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        random_state=SEED,
        class_weight="balanced",
        n_estimators=500,
        min_samples_leaf=2,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=SEED
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=SEED
    ),
}

if XGBOOST_AVAILABLE:
    models_base["XGBoost"] = XGBClassifier(
        random_state=SEED,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9
    )

if LIGHTGBM_AVAILABLE:
    models_base["LightGBM"] = LGBMClassifier(
        random_state=SEED,
        class_weight="balanced",
        n_estimators=300,
        learning_rate=0.05
    )

resultados_base = []
objetos_base = {}

for nombre, model in models_base.items():
    print("=" * 80)
    print("Entrenant model base:", nombre)

    try:
        res = entrenar_y_evaluar_thresholds(
            model,
            X_train_pca,
            y_train,
            X_val_pca,
            y_val,
            THRESHOLDS
        )

        objetos_base[nombre] = res

        resultados_base.append({
            "model": nombre,
            "threshold_f1": res["best_t_f1"],
            "val_f1_best": res["best_row_f1"]["val_f1_alzheimer"],
            "val_recall_f1": res["best_row_f1"]["val_recall_alzheimer"],
            "val_precision_f1": res["best_row_f1"]["val_precision_alzheimer"],
            "threshold_f2": res["best_t_f2"],
            "val_f2_best": res["best_row_f2"]["val_f2_alzheimer"],
            "val_recall_f2": res["best_row_f2"]["val_recall_alzheimer"],
            "val_precision_f2": res["best_row_f2"]["val_precision_alzheimer"],
        })

    except Exception as e:
        print("Error amb el model", nombre, ":", e)

df_base = pd.DataFrame(resultados_base).sort_values("val_f2_best", ascending=False)
display(df_base)


In [ ]:
# ==========================================
# 9. Visualització de la comparació inicial
# ==========================================

fig, ax = plt.subplots(figsize=(9, 5))

df_plot = df_base.sort_values("val_f2_best", ascending=True)

ax.barh(df_plot["model"], df_plot["val_f2_best"], alpha=0.7, label="Millor F2 en validació")
ax.barh(df_plot["model"], df_plot["val_f1_best"], alpha=0.4, label="Millor F1 en validació")

ax.set_xlabel("Score")
ax.set_title("Comparació inicial de models en validació")
ax.legend()
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()


## Cerca d'hiperparàmetres

A continuació es fa una cerca controlada d'hiperparàmetres. Per evitar fer una cerca massa gran, es proven només algunes combinacions raonables per a cada model.

La selecció de la millor configuració es fa amb el conjunt de validació, i es guarden dues opcions:

- millor configuració segons F1;
- millor configuració segons F2.


In [ ]:
# ==========================================
# 10. Definició de models i espais d'hiperparàmetres
# ==========================================

def crear_model(nombre, params):
    """
    Crea un model de classificació a partir del seu nom i dels hiperparàmetres indicats.

    Paràmetres
    ----------
    nombre : str
        Nom del model.
    params : dict
        Hiperparàmetres del model.

    Retorna
    -------
    object
        Instància del model.
    """
    if nombre == "Logistic Regression":
        return LogisticRegression(
            random_state=SEED,
            class_weight="balanced",
            max_iter=5000,
            **params
        )

    if nombre == "SVM RBF":
        return SVC(
            random_state=SEED,
            class_weight="balanced",
            probability=True,
            **params
        )

    if nombre == "Random Forest":
        return RandomForestClassifier(
            random_state=SEED,
            class_weight="balanced",
            n_jobs=-1,
            **params
        )

    if nombre == "Extra Trees":
        return ExtraTreesClassifier(
            random_state=SEED,
            class_weight="balanced",
            n_jobs=-1,
            **params
        )

    if nombre == "Gradient Boosting":
        return GradientBoostingClassifier(
            random_state=SEED,
            **params
        )

    if nombre == "HistGradientBoosting":
        return HistGradientBoostingClassifier(
            random_state=SEED,
            **params
        )

    if nombre == "XGBoost":
        return XGBClassifier(
            random_state=SEED,
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            **params
        )

    if nombre == "LightGBM":
        return LGBMClassifier(
            random_state=SEED,
            class_weight="balanced",
            verbose=-1,
            **params
        )

    raise ValueError(f"Model no reconegut: {nombre}")


param_grids = {
    "Logistic Regression": {
        "C": [0.1, 1.0, 10.0],
        "solver": ["lbfgs"]
    },
    "SVM RBF": {
        "C": [0.1, 1.0, 10.0],
        "gamma": ["scale", 0.01, 0.1]
    },
    "Random Forest": {
        "n_estimators": [300, 500],
        "max_depth": [None, 5, 10],
        "min_samples_leaf": [1, 2, 5],
        "max_features": ["sqrt", 0.5]
    },
    "Extra Trees": {
        "n_estimators": [300, 500],
        "max_depth": [None, 5, 10],
        "min_samples_leaf": [1, 2, 5],
        "max_features": ["sqrt", 0.5]
    },
    "Gradient Boosting": {
        "n_estimators": [100, 300],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [2, 3],
        "subsample": [0.8, 1.0]
    },
    "HistGradientBoosting": {
        "learning_rate": [0.03, 0.05, 0.1],
        "max_iter": [100, 300],
        "max_leaf_nodes": [15, 31],
        "l2_regularization": [0.0, 0.1]
    }
}

if XGBOOST_AVAILABLE:
    param_grids["XGBoost"] = {
        "n_estimators": [100, 300],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    }

if LIGHTGBM_AVAILABLE:
    param_grids["LightGBM"] = {
        "n_estimators": [100, 300],
        "learning_rate": [0.03, 0.05, 0.1],
        "num_leaves": [15, 31],
        "min_child_samples": [5, 10, 20]
    }

print("Models amb cerca d'hiperparàmetres:")
for k, v in param_grids.items():
    print(k, "— combinacions:", len(list(ParameterGrid(v))))


In [ ]:
# ==========================================
# 11. Cerca manual d'hiperparàmetres amb validació
# ==========================================

def buscar_hiperparametros_validacion(nombre_modelo, param_grid, max_combinaciones=None):
    """
    Prova diferents combinacions d'hiperparàmetres i avalua els llindars en validació.

    La funció retorna una taula amb el millor F1 i F2 de validació per a cada combinació.

    Paràmetres
    ----------
    nombre_modelo : str
        Nom del model.
    param_grid : dict
        Diccionari amb els hiperparàmetres a provar.
    max_combinaciones : int or None
        Nombre màxim de combinacions a provar. Si és None, es proven totes.

    Retorna
    -------
    tuple
        DataFrame de resultats i diccionari amb els objectes entrenats.
    """
    combinaciones = list(ParameterGrid(param_grid))

    if max_combinaciones is not None and len(combinaciones) > max_combinaciones:
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(combinaciones), size=max_combinaciones, replace=False)
        combinaciones = [combinaciones[i] for i in idx]

    rows = []
    objetos = {}

    for i, params in enumerate(combinaciones):
        print(f"{nombre_modelo}: combinació {i+1}/{len(combinaciones)} — {params}")

        try:
            model = crear_model(nombre_modelo, params)

            res = entrenar_y_evaluar_thresholds(
                model,
                X_train_pca,
                y_train,
                X_val_pca,
                y_val,
                THRESHOLDS
            )

            key = f"{nombre_modelo}_{i}"
            objetos[key] = res

            row = {
                "model": nombre_modelo,
                "key": key,
                "params": params,
                "threshold_f1": res["best_t_f1"],
                "val_f1_best": res["best_row_f1"]["val_f1_alzheimer"],
                "val_recall_f1": res["best_row_f1"]["val_recall_alzheimer"],
                "val_precision_f1": res["best_row_f1"]["val_precision_alzheimer"],
                "threshold_f2": res["best_t_f2"],
                "val_f2_best": res["best_row_f2"]["val_f2_alzheimer"],
                "val_recall_f2": res["best_row_f2"]["val_recall_alzheimer"],
                "val_precision_f2": res["best_row_f2"]["val_precision_alzheimer"],
            }
            rows.append(row)

        except Exception as e:
            print("Error:", e)

    return pd.DataFrame(rows), objetos


todos_resultados_search = []
todos_objetos_search = {}

# Limitem algunes cerques per evitar temps d'execució excessius.
MAX_COMBINACIONES_POR_MODELO = {
    "Logistic Regression": None,
    "SVM RBF": None,
    "Random Forest": None,
    "Extra Trees": None,
    "Gradient Boosting": None,
    "HistGradientBoosting": None,
    "XGBoost": 30,
    "LightGBM": 30,
}

for nombre_modelo, grid in param_grids.items():
    print("\n" + "=" * 100)
    print("Cerca d'hiperparàmetres:", nombre_modelo)
    print("=" * 100)

    df_model, obj_model = buscar_hiperparametros_validacion(
        nombre_modelo,
        grid,
        max_combinaciones=MAX_COMBINACIONES_POR_MODELO.get(nombre_modelo)
    )

    todos_resultados_search.append(df_model)
    todos_objetos_search.update(obj_model)

df_search = pd.concat(todos_resultados_search, ignore_index=True)
display(df_search.sort_values("val_f2_best", ascending=False).head(20))


In [ ]:
# ==========================================
# 12. Comparació dels millors hiperparàmetres per model
# ==========================================

best_f1_por_modelo = (
    df_search
    .sort_values(["model", "val_f1_best", "val_recall_f1", "val_precision_f1"], ascending=[True, False, False, False])
    .groupby("model")
    .head(1)
    .reset_index(drop=True)
)

best_f2_por_modelo = (
    df_search
    .sort_values(["model", "val_f2_best", "val_recall_f2", "val_precision_f2"], ascending=[True, False, False, False])
    .groupby("model")
    .head(1)
    .reset_index(drop=True)
)

print("Millor configuració segons F1 per model:")
display(best_f1_por_modelo[["model", "threshold_f1", "val_f1_best", "val_recall_f1", "val_precision_f1", "params"]])

print("Millor configuració segons F2 per model:")
display(best_f2_por_modelo[["model", "threshold_f2", "val_f2_best", "val_recall_f2", "val_precision_f2", "params"]])


In [ ]:
# ==========================================
# 13. Visualització abans/després de la cerca
# ==========================================

df_base_simple = df_base[["model", "val_f1_best", "val_f2_best"]].copy()
df_base_simple["tipus"] = "Model base"

df_best_simple = best_f2_por_modelo[["model", "val_f1_best", "val_f2_best"]].copy()
df_best_simple["tipus"] = "Millor configuració"

df_comparacio_search = pd.concat([df_base_simple, df_best_simple], ignore_index=True)

display(df_comparacio_search.sort_values(["model", "tipus"]))

plt.figure(figsize=(10, 5))

models_order = best_f2_por_modelo.sort_values("val_f2_best")["model"].tolist()
y_pos = np.arange(len(models_order))

base_vals = [
    df_base.loc[df_base["model"] == m, "val_f2_best"].values[0] if m in df_base["model"].values else np.nan
    for m in models_order
]
best_vals = [
    best_f2_por_modelo.loc[best_f2_por_modelo["model"] == m, "val_f2_best"].values[0]
    for m in models_order
]

plt.barh(y_pos - 0.2, base_vals, height=0.4, label="Base")
plt.barh(y_pos + 0.2, best_vals, height=0.4, label="Hiperparametritzat")

plt.yticks(y_pos, models_order)
plt.xlabel("F2 en validació")
plt.title("Comparació de models base i models hiperparametritzats")
plt.legend()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## Avaluació final en test

En aquesta secció s'avaluen sobre el conjunt de test els millors models seleccionats en validació. Es generen dues comparacions:

- millors models segons F1;
- millors models segons F2.

Aquesta avaluació és final: no es tornen a ajustar hiperparàmetres ni llindars amb el test.


In [ ]:
# ==========================================
# 14. Funció d'avaluació final en test
# ==========================================

def evaluar_modelo_en_test(nombre, key, criterio):
    """
    Avalua un model seleccionat sobre el conjunt de test.

    Paràmetres
    ----------
    nombre : str
        Nom del model.
    key : str
        Clau de l'objecte entrenat dins del diccionari `todos_objetos_search`.
    criterio : str
        Criteri de selecció del llindar: 'F1' o 'F2'.

    Retorna
    -------
    dict
        Resultats finals sobre el conjunt de test.
    """
    res = todos_objetos_search[key]
    model = res["model"]

    scores_test = get_scores_model(model, X_test_pca)

    if criterio == "F1":
        threshold = res["best_t_f1"]
    elif criterio == "F2":
        threshold = res["best_t_f2"]
    else:
        raise ValueError("criterio ha de ser 'F1' o 'F2'.")

    preds_test = (scores_test >= threshold).astype(int)

    row = {
        "model": nombre,
        "criteri": f"Millor {criterio} en validació",
        "threshold": threshold,
        "varianza_objetivo_pca": VARIANZA_OBJETIVO_PCA,
        "n_componentes_pca": N_COMPONENTS_PCA,
        "varianza_explicada_pca": varianza_acumulada[-1],
    }
    row.update(metricas_binarias(y_test, preds_test, prefix="test_"))

    return row, preds_test, scores_test


resultados_test = []

predicciones_test = {}
scores_test_dict = {}

for _, row in best_f1_por_modelo.iterrows():
    res_row, preds, scores = evaluar_modelo_en_test(row["model"], row["key"], "F1")
    resultados_test.append(res_row)
    predicciones_test[(row["model"], "F1")] = preds
    scores_test_dict[(row["model"], "F1")] = scores

for _, row in best_f2_por_modelo.iterrows():
    res_row, preds, scores = evaluar_modelo_en_test(row["model"], row["key"], "F2")
    resultados_test.append(res_row)
    predicciones_test[(row["model"], "F2")] = preds
    scores_test_dict[(row["model"], "F2")] = scores

df_resultados_test = pd.DataFrame(resultados_test)

display(
    df_resultados_test
    .sort_values(["criteri", "test_f2_alzheimer"], ascending=[True, False])
)


In [ ]:
# ==========================================
# 15. Millors models finals segons F1 i F2 en test
# ==========================================

print("Millors resultats en TEST segons F1:")
display(
    df_resultados_test
    .sort_values("test_f1_alzheimer", ascending=False)
    .head(10)
)

print("Millors resultats en TEST segons F2:")
display(
    df_resultados_test
    .sort_values("test_f2_alzheimer", ascending=False)
    .head(10)
)

print("Millors resultats en TEST segons recall:")
display(
    df_resultados_test
    .sort_values("test_recall_alzheimer", ascending=False)
    .head(10)
)


In [ ]:
# ==========================================
# 16. Matrius de confusió dels millors models
# ==========================================

best_test_f1 = df_resultados_test.sort_values("test_f1_alzheimer", ascending=False).iloc[0]
best_test_f2 = df_resultados_test.sort_values("test_f2_alzheimer", ascending=False).iloc[0]

for best_row in [best_test_f1, best_test_f2]:
    model_name = best_row["model"]
    criteri = "F1" if "F1" in best_row["criteri"] else "F2"
    preds = predicciones_test[(model_name, criteri)]

    print("=" * 80)
    print(best_row["model"], "—", best_row["criteri"])
    print("=" * 80)

    print(classification_report(
        y_test,
        preds,
        target_names=["Sa", "Alzheimer"],
        zero_division=0
    ))

    plot_confusion_catala(
        y_test,
        preds,
        f"TEST — {model_name} — {best_row['criteri']} — threshold={best_row['threshold']:.2f}"
    )


In [ ]:
# ==========================================
# 17. Corbes ROC i Precision-Recall dels millors models
# ==========================================

models_to_plot = [
    (best_test_f1["model"], "F1"),
    (best_test_f2["model"], "F2"),
]

plt.figure(figsize=(7, 5))

for model_name, criteri in models_to_plot:
    scores = scores_test_dict[(model_name, criteri)]
    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{model_name} ({criteri}) — AUC={roc_auc:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Taxa de falsos positius")
plt.ylabel("Taxa de verdaders positius")
plt.title("Corba ROC en test")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


plt.figure(figsize=(7, 5))

for model_name, criteri in models_to_plot:
    scores = scores_test_dict[(model_name, criteri)]
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    plt.plot(recall, precision, label=f"{model_name} ({criteri}) — AP={ap:.3f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Corba Precision-Recall en test")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 18. Anàlisi de probabilitats i casos extrems
# ==========================================

# Agafem el millor model segons F2, perquè és el criteri més orientat a reduir falsos negatius.
model_name = best_test_f2["model"]
criteri = "F2"
scores = scores_test_dict[(model_name, criteri)]
preds = predicciones_test[(model_name, criteri)]

df_test_analysis = pd.DataFrame({
    "OASISID": groups_test,
    "y_true": y_test,
    "y_pred": preds,
    "prob_alzheimer": scores
})

df_test_analysis["tipus_error"] = "Correcte"
df_test_analysis.loc[(df_test_analysis["y_true"] == 0) & (df_test_analysis["y_pred"] == 1), "tipus_error"] = "Fals positiu"
df_test_analysis.loc[(df_test_analysis["y_true"] == 1) & (df_test_analysis["y_pred"] == 0), "tipus_error"] = "Fals negatiu"

print("Distribució dels errors:")
display(df_test_analysis["tipus_error"].value_counts())

print("Falsos positius amb probabilitat més alta:")
display(df_test_analysis[df_test_analysis["tipus_error"] == "Fals positiu"].sort_values("prob_alzheimer", ascending=False).head(10))

print("Falsos negatius:")
display(df_test_analysis[df_test_analysis["tipus_error"] == "Fals negatiu"].sort_values("prob_alzheimer", ascending=False))

plt.figure(figsize=(8, 5))
for label, name in [(0, "Sa"), (1, "Alzheimer")]:
    plt.hist(
        df_test_analysis.loc[df_test_analysis["y_true"] == label, "prob_alzheimer"],
        bins=20,
        alpha=0.6,
        label=name
    )

plt.axvline(best_test_f2["threshold"], linestyle="--", label=f"Threshold F2={best_test_f2['threshold']:.2f}")
plt.xlabel("Probabilitat predita d'Alzheimer")
plt.ylabel("Nombre de mostres")
plt.title(f"Distribució de probabilitats — {model_name}")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Notes finals

Aquesta notebook permet veure si afegir més models clàssics i cerca d'hiperparàmetres millora el Random Forest inicial. Si cap model supera clarament el Random Forest, això també és un resultat útil: indica que la limitació principal pot estar més relacionada amb les dades, el senyal disponible o la definició de l'etiqueta que no pas amb l'elecció d'un classificador concret.
